<a href="https://colab.research.google.com/github/yehiaelsalmi/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers pypdf python-dotenv pandas

## 1. Setup

In [ ]:
import os, re, time
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

PDF_URL = "https://www.tutorialspoint.com/machine_learning/machine_learning_tutorial.pdf"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 3
LLM_MODEL = None
REFUSAL = "I don't know based on the provided document."

In [ ]:
def load_groq_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GROQ_API_KEY")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = load_groq_key() or ""
assert os.environ["GROQ_API_KEY"], "Add GROQ_API_KEY as a secret or in a .env file"

def pick_groq_model(requested=None):
    from groq import Groq
    available = {m.id for m in Groq(api_key=os.environ["GROQ_API_KEY"]).models.list().data}
    if requested and requested in available:
        return requested
    skip = ("whisper", "guard", "tts", "playai", "orpheus", "distil", "compound")
    chat_models = [m for m in available if not any(s in m for s in skip)]
    rank = lambda m: (0 if "llama" in m else 1 if "gpt-oss" in m else 2, m)
    if not chat_models:
        raise RuntimeError(f"No chat model found. Available: {sorted(available)}")
    return sorted(chat_models, key=rank)[0]

LLM_MODEL = pick_groq_model(LLM_MODEL)


llm = ChatGroq(model=LLM_MODEL, temperature=0)
llm.invoke("Reply with OK")
print(f"Using Groq model: {LLM_MODEL}")

Using Groq model: openai/gpt-oss-120b


In [ ]:
pages = PyPDFLoader(PDF_URL).load()
print(f"Loaded {len(pages)} pages")

def is_toc_page(text):
    """Table-of-contents pages contain many dot leaders such as '.......... 26'."""
    return len(re.findall(r"\.{5,}", text)) >= 5

def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()

toc_pages = [p.metadata.get("page", 0) + 1 for p in pages if is_toc_page(p.page_content)]
clean_pages = [
    Document(page_content=clean_text(p.page_content), metadata=p.metadata)
    for p in pages
    if not is_toc_page(p.page_content) and p.page_content.strip()
]

print(f"Table-of-contents pages removed: {toc_pages}")
print(f"Pages kept: {len(clean_pages)}")

Loaded 35 pages
Table-of-contents pages removed: [3, 4]
Pages kept: 33


## 3. Split into chunks

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],
)

raw_chunks = splitter.split_documents(pages)
chunks = splitter.split_documents(clean_pages)

print(f"Baseline chunks: {len(raw_chunks)} | Clean chunks: {len(chunks)}")
print(f"Average clean chunk length: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} characters")

Baseline chunks: 66 | Clean chunks: 74
Average clean chunk length: 555 characters


## 4. Embeddings and vector store

Each chunk becomes a 384-number vector that captures its meaning. FAISS stores the vectors and finds the ones closest to a question's vector.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

raw_store = FAISS.from_documents(raw_chunks, embeddings)
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

print(f"Vectors in baseline index: {raw_store.index.ntotal}")
print(f"Vectors in clean index:    {vectorstore.index.ntotal}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectors in baseline index: 66
Vectors in clean index:    74


In [ ]:
prompt = ChatPromptTemplate.from_template("""You are an assistant that answers questions about a document.
Use ONLY the context below. If the answer is not in the context, reply exactly:
"{refusal}"
Keep the answer concise.

Context:
{context}

Question: {question}

Answer:""")

answer_chain = prompt | llm | StrOutputParser()

def format_docs(docs):
    return "\n\n".join(f"[page {d.metadata.get('page', 0) + 1}] {d.page_content}" for d in docs)

def ask(question, store=vectorstore, k=TOP_K):

    docs = store.similarity_search(question, k=k)
    answer = answer_chain.invoke({"context": format_docs(docs), "question": question, "refusal": REFUSAL})
    sources = sorted({d.metadata.get("page", 0) + 1 for d in docs})
    return {"answer": answer, "docs": docs, "sources": sources}

In [ ]:
demo_questions = [
    "What are the main categories of machine learning?",
    "Explain supervised learning with an example.",
    "What is the difference between classification and regression?",
]

for q in demo_questions:
    r = ask(q)
    print(f"Q: {q}\nA: {r['answer']}\nSources: pages {r['sources']}\n")

Q: What are the main categories of machine learning?
A: I don't know based on the provided document.
Sources: pages [5, 11, 33]

Q: Explain supervised learning with an example.
A: Supervised learning is a type of machine learning where the model is trained on labeled examples—pairs of inputs and known outputs. For instance, you might give the computer data points such as “for feature value x₁ the output is y₁, for x₂ the output is y₂, …”. The algorithm learns the empirical relationship between x and y, and after training it can predict the output Y for a new input X. Its performance is then checked using test data with known true values, similar to how a child learns to walk by being guided and demonstrated before walking independently.
Sources: pages [6, 11, 13]

Q: What is the difference between classification and regression?
A: Regression predicts a continuous numeric value Y for a given input X by learning an empirical relationship, whereas classification assigns an object to one o

In [ ]:
eval_set = [
    {"question": "What are the main categories of machine learning?", "evidence": "broadly categorized"},
    {"question": "What analogy does the tutorial use to explain supervised learning?", "evidence": "training a child to walk"},
    {"question": "Which example is used to explain predicting a value with supervised learning?", "evidence": "housing price prediction"},
    {"question": "How does the tutorial explain classification with an example?", "evidence": "short, medium and long"},
    {"question": "Why has AI become so widely used today, according to the tutorial?", "evidence": "computing resources"},
    {"question": "Which statistical techniques did early AI programs use?", "evidence": "statistical techniques"},
    {"question": "Why were more advanced methods such as deep learning developed?", "evidence": "deep learning are hence developed"},
    {"question": "What questions does the tutorial raise about tools and platforms for machine learning?", "evidence": "platforms support"},
    {"question": "What is the capital of France?", "evidence": None},
    {"question": "What was TutorialsPoint's revenue in 2023?", "evidence": None},
    {"question": "How does the attention mechanism in transformers work?", "evidence": None},
]

def norm(s):
    return re.sub(r"\s+", " ", s).lower()

for item in eval_set:
    if item["evidence"]:
        found = any(norm(item["evidence"]) in norm(c.page_content) for c in chunks)
        print("OK     " if found else "MISSING", item["evidence"])

OK      broadly categorized
OK      training a child to walk
OK      housing price prediction
OK      short, medium and long
OK      computing resources
OK      statistical techniques
OK      deep learning are hence developed
OK      platforms support


In [ ]:
def retrieval_metrics(store, k):
    hits, reciprocal_ranks = [], []
    for item in eval_set:
        if not item["evidence"]:
            continue
        docs = store.similarity_search(item["question"], k=k)
        rank = next((i + 1 for i, d in enumerate(docs) if norm(item["evidence"]) in norm(d.page_content)), None)
        hits.append(rank is not None)
        reciprocal_ranks.append(1 / rank if rank else 0)
    return sum(hits) / len(hits), sum(reciprocal_ranks) / len(reciprocal_ranks)

configs = [
    ("Baseline (raw pages)", raw_store, 3),
    ("Clean pages", vectorstore, 3),
    ("Clean pages", vectorstore, 5),
]

rows = []
for name, store, k in configs:
    hit_rate, mrr = retrieval_metrics(store, k)
    rows.append({"setup": name, "k": k, "hit_rate": round(hit_rate, 2), "mrr": round(mrr, 2)})

retrieval_results = pd.DataFrame(rows)
retrieval_results

,setup,k,hit_rate,mrr
0,Baseline (raw pages),3,0.38,0.38
1,Clean pages,3,0.50,0.35
2,Clean pages,5,0.50,0.35


In [ ]:
def is_refusal(answer):
    a = answer.lower()
    return "don't know" in a or "do not know" in a

rows = []
for item in eval_set:
    if item["evidence"] is None:
        r = ask(item["question"])
        rows.append({"question": item["question"], "answer": r["answer"], "refused": is_refusal(r["answer"])})
        time.sleep(3)

refusal_results = pd.DataFrame(rows)
print(f"Correct refusals: {refusal_results['refused'].sum()} / {len(refusal_results)}")
refusal_results

Correct refusals: 3 / 3


,question,answer,refused
0,What is the capital of France?,I don't know based on the provided document.,True
1,What was TutorialsPoint's revenue in 2023?,I don't know based on the provided document.,True
2,How does the attention mechanism in transforme...,I don't know based on the provided document.,True
